In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# -------------------------------
# Dense GCN layer (same as baseline)
# -------------------------------
class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x, A):
        # x: (B, N, C_in); A: (N, N)
        x_prop = torch.matmul(A, x)
        return self.lin(x_prop)

# -------------------------------
# Fixed grid adjacency (normalized) — used only to initialize A
# -------------------------------
def make_grid_adjacency_norm(h=7, w=7, device="cpu"):
    N = h * w
    A = torch.zeros((N, N), dtype=torch.float32)
    def idx(r, c): return r * w + c

    for r in range(h):
        for c in range(w):
            u = idx(r, c)
            A[u, u] = 1.0  # self-loop
            if r > 0:   A[u, idx(r - 1, c)] = 1.0
            if r < h-1: A[u, idx(r + 1, c)] = 1.0
            if c > 0:   A[u, idx(r, c - 1)] = 1.0
            if c < w-1: A[u, idx(r, c + 1)] = 1.0

    # D^{-1/2} A D^{-1/2}
    deg = A.sum(dim=1)
    D_inv_sqrt = torch.diag(torch.pow(deg + 1e-8, -0.5))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return A_norm.to(device)

# -------------------------------
# Learnable adjacency, softmax-normalized each forward
# -------------------------------
class MobileNetV3_GCN_LearnAdj_A2(nn.Module):
    def __init__(self, num_classes, version="large", pretrained=True, gcn_hidden=128,  # Reduced hidden units
                 freeze_backbone=False, device="cpu", symmetrize=True, init_scale=4.0):
        super().__init__()

        # --- Backbone ---
        if version == "large":
            backbone = models.mobilenet_v3_large(
                weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
            )
            feat_dim = 960
        else:
            backbone = models.mobilenet_v3_small(
                weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
            )
            feat_dim = 576

        self.features = backbone.features
        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False

        # --- Graph / GCN ---
        self.h, self.w = 7, 7
        self.N = self.h * self.w
        self.gcn1 = DenseGCNLayer(feat_dim, gcn_hidden)  # 128 hidden units
        self.gcn2 = DenseGCNLayer(gcn_hidden, gcn_hidden)  # 128 hidden units

        # Add another GCN layer
        self.gcn3 = DenseGCNLayer(gcn_hidden, gcn_hidden)  # 128 hidden units

        # Increased dropout to reduce overfitting
        self.dropout = nn.Dropout(p=0.4)  # Dropout at 0.4
        self.cls = nn.Linear(gcn_hidden, num_classes)

        # Learnable adjacency logits
        with torch.no_grad():
            A_init = make_grid_adjacency_norm(self.h, self.w, device=device)
            A_init = A_init / (A_init.sum(dim=-1, keepdim=True) + 1e-12)
            A_logits_init = torch.log(A_init + 1e-8) * init_scale
        self.A_logits = nn.Parameter(A_logits_init)

        self.symmetrize = symmetrize

    def _get_A_norm(self):
        A = F.softmax(self.A_logits, dim=-1)
        if self.symmetrize:
            A = 0.5 * (A + A.transpose(0, 1))
            A = A / (A.sum(dim=-1, keepdim=True) + 1e-12)
        return A

    def forward(self, x):
        f = self.features(x)
        B, C, H, W = f.shape
        assert (H, W) == (self.h, self.w), f"Expected {(self.h,self.w)}, got {(H,W)}"
        f = f.view(B, C, H * W).transpose(1, 2)

        A_norm = self._get_A_norm()
        z = F.relu(self.gcn1(f, A_norm))
        z = F.relu(self.gcn2(z, A_norm))

        # Add the third GCN layer to improve learning
        z = F.relu(self.gcn3(z, A_norm))

        zg = z.mean(dim=1)
        out = self.cls(self.dropout(zg))  # Apply dropout before final classification
        return out

    # Optional regularizer
    def adj_regularizer(self, lam_entropy=0.0, lam_deviation=0.0):
        reg = 0.0
        if lam_entropy > 0:
            A = F.softmax(self.A_logits, dim=-1)
            ent = (-A * (A.add(1e-12).log())).sum(dim=-1).mean()
            reg = reg - lam_entropy * ent
        if lam_deviation > 0:
            reg = reg + lam_deviation * (self.A_logits ** 2).mean()
        return reg

num_classes = len(class_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MobileNetV3_GCN_LearnAdj_A2(
    num_classes=num_classes,
    version="large",
    pretrained=True,
    gcn_hidden=128,  # Reduced hidden units to 128
    freeze_backbone=False,
    device=device,
    symmetrize=True,   # set False if you prefer purely row-stochastic A
    init_scale=4.0     # larger => starts closer to the grid structure
).to(device)

In [ ]:
model_name = "TeaNet"

print(f"=== {model_name} ===")
print(f"\nParameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Classes    : {len(class_names)}")